# AdaBoost Regressor - California Housing Dataset
### Target: Predict median_house_value

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## Step 1 - Load Dataset

In [ ]:
df = pd.read_csv('california_housing.csv')
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Missing values check
df.isnull().sum()

## Step 2 - EDA

In [ ]:
# Target distribution
plt.figure(figsize=(8, 4))
sns.histplot(df['median_house_value'], bins=50, kde=True, color='steelblue')
plt.title('House Price Distribution')
plt.xlabel('Median House Value ($)')
plt.show()

In [ ]:
# ocean_proximity categories
print(df['ocean_proximity'].value_counts())

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.show()

## Step 3 - Preprocessing

In [ ]:
# Fill missing values
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())

# Encode ocean_proximity
le = LabelEncoder()
df['ocean_proximity'] = le.fit_transform(df['ocean_proximity'])

print("Missing values after fix:", df.isnull().sum().sum())
df.head()

## Step 4 - Train Test Split

In [ ]:
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape)
print("Test size: ", X_test.shape)

## Step 5 - Train AdaBoost Regressor

In [ ]:
ada_reg = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(max_depth=4),
    n_estimators=100,
    learning_rate=0.1,
    loss='linear',
    random_state=42
)

ada_reg.fit(X_train, y_train)
print("Model trained successfully!")

## Step 6 - Evaluation

In [ ]:
y_pred = ada_reg.predict(X_test)

r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R2 Score : {r2:.4f}")
print(f"MAE      : ${mae:,.0f}")
print(f"RMSE     : ${rmse:,.0f}")

In [ ]:
# Actual vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.3, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Fit')
plt.xlabel('Actual House Value ($)')
plt.ylabel('Predicted House Value ($)')
plt.title('Actual vs Predicted')
plt.legend()
plt.show()

In [ ]:
# Residuals plot
residuals = y_test - y_pred
plt.figure(figsize=(8, 4))
plt.scatter(y_pred, residuals, alpha=0.3, color='orange')
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residuals Plot')
plt.show()

## Step 7 - Feature Importance

In [ ]:
feat_imp = pd.Series(ada_reg.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='viridis')
plt.title('Feature Importance')
plt.xlabel('Importance Score')
plt.show()

print(feat_imp)

## Step 8 - Compare Loss Functions

In [ ]:
print(f"{'Loss':<15} {'R2':>8} {'MAE':>12} {'RMSE':>12}")
print("-" * 50)

for loss in ['linear', 'square', 'exponential']:
    model = AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=4),
        n_estimators=100,
        learning_rate=0.1,
        loss=loss,
        random_state=42
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    r2   = r2_score(y_test, preds)
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    print(f"{loss:<15} {r2:>8.4f} {mae:>12,.0f} {rmse:>12,.0f}")

## Step 9 - Effect of n_estimators

In [ ]:
estimator_range = [10, 50, 100, 200, 300]
r2_scores = []

for n in estimator_range:
    model = AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=4),
        n_estimators=n,
        learning_rate=0.1,
        random_state=42
    )
    model.fit(X_train, y_train)
    r2_scores.append(r2_score(y_test, model.predict(X_test)))

plt.figure(figsize=(8, 4))
plt.plot(estimator_range, r2_scores, marker='o', color='green', linewidth=2)
plt.xlabel('n_estimators')
plt.ylabel('R2 Score')
plt.title('n_estimators vs R2 Score')
plt.grid(True)
plt.show()